# Projecting Observation Records onto the Calendar

This notebook demonstrates `solsys_code/observation_projector.py` and
`solsys_code/management/commands/project_observation_calendar.py` -- the `post_save`
receiver that draws and narrows an LCO/SOAR `ObservationRecord`'s own `CalendarEvent`
with no operator command, and the sweep that is its backstop for bulk write paths
(`QuerySet.update()`, `bulk_create()`, `backfill_lco_observations`, and the one-time
observed-telescope lookup for a newly observed record).

It demonstrates:

- The receiver narrowing one throwaway record's event through `[Q]` -> `[S]` -> `[O]`
  with no command run at all (rolled back at the end -- nothing here persists)
- The **one-time takeover** (D-19): the first sweep after this phase re-titles and
  re-links every legacy LCO/SOAR calendar entry an earlier, now-retired sync command
  had already created, while the `RUN:`-prefixed (campaign reconciler), `GEM:`-prefixed
  (Gemini submission-echo) and blank-url (hand-entered/classical) event families are
  proven byte-identical before and after
- A second sweep over the now-converged corpus reporting zero created/updated/
  site-lookups -- convergence, not silence
- The whole real LCO/SOAR corpus reconciled against the calendar, not a sample, with
  every unprojectable record enumerated by reason
- The SCHED-06 baseline over the pending `KEY2026B-004` records -- the evidence this
  notebook's own re-execution, after real observing nights pass, will close

This notebook lives in `pre_executed/` because it is **DB-dependent** and makes **real
LCO Observation Portal calls** (the one-time observed-telescope lookup, D-07/D-08), so
it is **NOT** run during Sphinx/CI/ReadTheDocs builds, per `docs/notebooks/README.md`.

**This notebook is written for the developer database (`src/fomo_db.sqlite3`).** The
whole point of the SCHED-06 section below is a claim about *this* database's real
records over real time -- a discarded copy could not support that claim. It also
honours a `FOMO_DATABASE_PATH` override, so a re-execution can be routed to a fresh
clone when the developer database is holding evidence a sweep would spend (see "The
one-time takeover" below for the un-swept-clone rule that routing requires). **This
commit's run used a scratch copy, not the developer database** -- see the "Django
setup" cell's own printed output for the authoritative answer to which database any
given execution used. The one part of this notebook that does not persist is the
receiver demo immediately below, which runs inside a transaction that is rolled back
on purpose.


## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's
location (`docs/notebooks/pre_executed/` -- three levels under the repo root, so
`parents[2]` gives the repo root) and to allow synchronous ORM calls inside Jupyter's
async event loop, matching this repo's other `pre_executed/` notebooks.

This notebook normally runs against the real developer database -- that is still the
point, since the takeover and the per-family reconciliation below are about real
records -- and it now also honours a `FOMO_DATABASE_PATH` override, so a re-execution
can be routed to a scratch copy when the developer database is holding evidence that a
sweep would spend. This commit's own run was routed to a copy for exactly that reason:
the developer database is carrying the stale LCO events that G-34-2 left behind, which
the next real `updatestatus` run must repair through the receiver alone, for SCHED-06.


In [1]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
if not (repo_root_path / 'manage.py').exists():
    raise RuntimeError(f'No manage.py at {repo_root_path}; run Jupyter from docs/notebooks/pre_executed/')
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

dev_db_path = repo_root_path / 'src' / 'fomo_db.sqlite3'
if not dev_db_path.exists():
    raise RuntimeError(f'No developer database at {dev_db_path}; run `python manage.py migrate` first.')

# An absolute FOMO_DATABASE_PATH override routes this notebook to a scratch copy instead
# of the developer database (see the framing above) -- an empty string is treated as
# unset, matching settings.py's own `or` semantics.
SCRATCH_DB_OVERRIDE = os.environ.get('FOMO_DATABASE_PATH') or None

django.setup()

# This notebook intentionally imports only the calendar-projection models/module below --
# never the ephemeris view/computation modules, which trigger a large one-time SPICE
# kernel download on first import.

from django.conf import settings as django_settings

resolved_db_name = django_settings.DATABASES['default']['NAME']
if SCRATCH_DB_OVERRIDE is None:
    assert resolved_db_name == str(dev_db_path), (
        f'This notebook is meant to run against the developer database itself -- resolved DB '
        f'{resolved_db_name!r} is not {str(dev_db_path)!r}.'
    )
    print(f'Django ready: settings module={os.environ["DJANGO_SETTINGS_MODULE"]!r}, repo_root={repo_root!r}')
    print(f'Resolved database: {resolved_db_name!r} (the developer database itself -- see framing above)')
else:
    # WR-08: compare resolved paths and raise rather than assert -- the previous guard's
    # first conjunct (`resolved_db_name == SCRATCH_DB_OVERRIDE`) is a tautology (Django's
    # own NAME setting is `os.getenv('FOMO_DATABASE_PATH')` verbatim), so the only real
    # check was a string inequality against an absolute dev_db_path. A relative override
    # (e.g. '../../src/fomo_db.sqlite3') passed that check while SQLite opened the
    # developer database itself -- and the very next cells run a real, non-transactional
    # takeover sweep against it. Comparing resolved (Path.resolve()) paths closes that
    # gap, and raising (not asserting, which python -O strips) keeps this a real guard.
    resolved_override = Path(resolved_db_name).resolve()
    if resolved_override == dev_db_path.resolve():
        raise RuntimeError(
            f'FOMO_DATABASE_PATH={SCRATCH_DB_OVERRIDE!r} resolves to the developer database '
            f'({dev_db_path}); point it at a scratch copy.'
        )
    print(f'Django ready: settings module={os.environ["DJANGO_SETTINGS_MODULE"]!r}, repo_root={repo_root!r}')
    print(f'Resolved database: {resolved_db_name!r} -- routed to a scratch copy, not the developer database.')

Django ready: settings module='src.fomo.settings', repo_root='/home/tlister/git/fomo_devel'
Resolved database: '/home/tlister/git/fomo_devel/tmp/34-07-fresh-clone.sqlite3' -- routed to a scratch copy, not the developer database.


## The receiver, demonstrated -- no operator command

A throwaway LCO `ObservationRecord` is created inside a transaction that is rolled
back at the end of this cell, using `NonSiderealTargetFactory` for its target -- FOMO
targets are Solar System minor bodies, never sidereal sources (CLAUDE.md). Nothing in
this cell persists past it: not the demo user, not the demo target, not the record,
and not its event.

Three saves, three narrowings, zero commands run:

1. **Creation** -- the event appears with a `[Q]` title and the request-window span
   from `parameters['start']`/`['end']`.
2. **A schedule-only save** (`scheduled_start`/`scheduled_end` populated, simulating
   the LCO scheduler placing the observation) -- the same event narrows to a `[S]`
   block span. The two schedule values below are assigned as the ISO-8601 strings the
   LCO portal returns, not as `datetime` objects, because that is literally what
   `BaseObservationFacility.update_observation_status()` does with
   `OCSFacility.get_observation_status()`'s payload. This is the path gap G-34-2 found
   broken: the receiver used to raise `AttributeError` inside `project_record()`, log
   `unprojectable`, and leave the event stale -- it no longer does.
3. **A terminal-success status save** (`status='COMPLETED'`) -- the title turns `[O]`.


In [2]:
from django.contrib.auth import get_user_model
from django.db import transaction
from tom_calendar.models import CalendarEvent
from tom_observations.models import ObservationRecord
from tom_targets.tests.factories import NonSiderealTargetFactory

from solsys_code.observation_projector import event_url, facility_for


class _RollbackDemo(Exception):
    """Raised on purpose at the end of the receiver demo to force a full rollback."""


demo_event_url = None
try:
    with transaction.atomic():
        demo_user = get_user_model().objects.create(username='observation-projector-demo-user')
        demo_target = NonSiderealTargetFactory.create(name='observation-projector-demo-target')

        record = ObservationRecord.objects.create(
            target=demo_target,
            user=demo_user,
            facility='LCO',
            observation_id='projector-demo-900001',
            status='PENDING',
            parameters={
                'proposal': 'PROJDEMO',
                'start': '2026-09-20T00:00:00',
                'end': '2026-09-21T00:00:00',
                'instrument_type': '2M0-SCICAM-MUSCAT',
            },
        )
        facility = facility_for(record)
        demo_event_url = event_url(record, facility)
        event = CalendarEvent.objects.get(url=demo_event_url)
        print('1. After creation (no operator command run):')
        print(f'   title={event.title!r}')
        print(f'   start={event.start_time.isoformat()}  end={event.end_time.isoformat()}')

        record.scheduled_start = '2026-09-20T03:00:00Z'
        record.scheduled_end = '2026-09-20T05:00:00Z'
        record.save()
        event.refresh_from_db()
        print()
        print('2. After a schedule-only save (still no operator command):')
        print(f'   assigned scheduled_start={record.scheduled_start!r} scheduled_end={record.scheduled_end!r}')
        print(f'   title={event.title!r}')
        print(f'   start={event.start_time.isoformat()}  end={event.end_time.isoformat()}')

        record.status = 'COMPLETED'
        record.save()
        event.refresh_from_db()
        print()
        print('3. After a terminal-success status save:')
        print(f'   title={event.title!r}')
        print(f'   start={event.start_time.isoformat()}  end={event.end_time.isoformat()}')

        raise _RollbackDemo
except _RollbackDemo:
    pass

print()
print('Rolled back -- nothing above persists:')
print('  demo record still in DB:', ObservationRecord.objects.filter(observation_id='projector-demo-900001').exists())
print('  demo event still in DB :', CalendarEvent.objects.filter(url=demo_event_url).exists())

No Profile found for observation-projector-demo-user. Creating Profile.


Target post save hook: observation-projector-demo-target created: True


Target post save hook: observation-projector-demo-target created: False


Observation change state hook: observation-projector-demo-target @ LCO from None to PENDING


Observation change state hook: observation-projector-demo-target @ LCO from PENDING to COMPLETED


1. After creation (no operator command run):
   title='[Q] 2m0 observation-projector-demo-target'
   start=2026-09-20T00:00:00+00:00  end=2026-09-21T00:00:00+00:00

2. After a schedule-only save (still no operator command):
   assigned scheduled_start='2026-09-20T03:00:00Z' scheduled_end='2026-09-20T05:00:00Z'
   title='[S] 2m0 observation-projector-demo-target'
   start=2026-09-20T03:00:00+00:00  end=2026-09-20T05:00:00+00:00

3. After a terminal-success status save:
   title='[O] 2m0 observation-projector-demo-target'
   start=2026-09-20T03:00:00+00:00  end=2026-09-20T05:00:00+00:00

Rolled back -- nothing above persists:
  demo record still in DB: False
  demo event still in DB : False


## The one-time takeover (D-19)

Every existing `CalendarEvent` is snapshotted now, before any write below, and
partitioned into four disjoint key families by url:

- **facility-url-keyed** -- this module's own namespace
  (`facility.get_observation_url(record.observation_id)`), including every event an
  earlier, now-retired sync command already created under the same url scheme
- **`RUN:`** -- the campaign reconciler's own namespace
  (`solsys_code/campaign_reconciler.py`)
- **`GEM:`** -- Gemini's submission-echo namespace
  (`sync_gemini_observation_calendar`)
- **blank-url** -- hand-entered and classically-scheduled entries with no url at all

`--dry-run` previews what the sweep would do without writing anything. The real sweep
that follows already performed its actual, one-time takeover against the developer
database when plan 34-04 first executed this notebook; a later re-execution shows the
same sweep run again, against whichever database it is pointed at -- for this commit, a
scratch copy (see "Django setup" above). A re-execution routed to a scratch copy must
start from a clone taken from an un-swept database, because a copy an earlier sweep
already converged produces a takeover that takes nothing over -- the cells below now
assert exactly that, so such a run aborts rather than committing empty evidence. This
run's own first sweep reported LCO created: 0, updated: 33, and re-titled 33 of 159
pre-existing facility-url-keyed events -- those are exactly the events the receiver
failed to narrow before plan 34-05's schedule-string fix, repaired here on the clone by
the sweep just as the operator's next `updatestatus` will repair them on the developer
database through the receiver alone. The second sweep reporting zeros for every
facility is still the convergence claim: once a database's events match the
projector's own vocabulary, re-running the sweep changes nothing.


In [3]:
from django.core.exceptions import ObjectDoesNotExist
from tom_calendar.models import CalendarEvent
from tom_observations.models import ObservationRecord

from solsys_code.observation_projector import PROJECTED_FACILITIES, event_url, facility_for


def snapshot_by_namespace():
    """Partition every CalendarEvent into the four disjoint key families above.

    Each event's snapshot key is (url, title, start_time, end_time, companion-row
    observation_record id, companion-row observation_group id) -- the exact tuple
    D-19 requires the RUN:/GEM:/blank-url families to be proven byte-identical on.
    """
    facility_urls = {
        event_url(r, facility_for(r)) for r in ObservationRecord.objects.filter(facility__in=PROJECTED_FACILITIES)
    }
    families = {'facility-url-keyed': {}, 'RUN:': {}, 'GEM:': {}, 'blank-url': {}}
    for calendar_event in CalendarEvent.objects.select_related('telescope_label_meta').all():
        try:
            meta = calendar_event.telescope_label_meta
            obs_record_id, obs_group_id = meta.observation_record_id, meta.observation_group_id
        except ObjectDoesNotExist:
            obs_record_id, obs_group_id = None, None
        key = (
            calendar_event.url,
            calendar_event.title,
            calendar_event.start_time,
            calendar_event.end_time,
            obs_record_id,
            obs_group_id,
        )
        if calendar_event.url in facility_urls:
            families['facility-url-keyed'][calendar_event.pk] = key
        elif calendar_event.url.startswith('RUN:'):
            families['RUN:'][calendar_event.pk] = key
        elif calendar_event.url.startswith('GEM:'):
            families['GEM:'][calendar_event.pk] = key
        elif not calendar_event.url:
            families['blank-url'][calendar_event.pk] = key
    return families


before_families = snapshot_by_namespace()
print('Every existing CalendarEvent, before any write in this notebook:')
for family_name, family in before_families.items():
    print(f'  {family_name:<20}: {len(family)} events')

Every existing CalendarEvent, before any write in this notebook:
  facility-url-keyed  : 159 events
  RUN:                : 72 events
  GEM:                : 0 events
  blank-url           : 10 events


In [4]:
from io import StringIO

from django.core.management import call_command

dry_run_stdout = StringIO()
call_command('project_observation_calendar', '--dry-run', stdout=dry_run_stdout, stderr=StringIO())
print('--dry-run preview (writes nothing):')
print(dry_run_stdout.getvalue().strip())

--dry-run preview (writes nothing):
Done (dry run). failed: 0 | LCO: created: 0, updated: 33, unchanged: 126, unprojectable: 0, site_lookups: 0, site_lookup_failed: 0 | SOAR: created: 0, updated: 0, unchanged: 0, unprojectable: 0, site_lookups: 0, site_lookup_failed: 0


In [5]:
first_sweep_stdout = StringIO()
first_sweep_stderr = StringIO()
call_command('project_observation_calendar', stdout=first_sweep_stdout, stderr=first_sweep_stderr)
first_sweep_summary = first_sweep_stdout.getvalue().strip()
print('First real sweep -- the one-time takeover:')
print(first_sweep_summary)
if first_sweep_stderr.getvalue():
    print()
    print('stderr (per-record site-lookup-failed lines, retried on the next sweep):')
    print(first_sweep_stderr.getvalue().strip())

after_families = snapshot_by_namespace()
print()
print('Every existing CalendarEvent, after the takeover sweep:')
for family_name, family in after_families.items():
    print(f'  {family_name:<20}: {len(family)} events')

First real sweep -- the one-time takeover:
Done. failed: 0 | LCO: created: 0, updated: 33, unchanged: 126, unprojectable: 0, site_lookups: 14, site_lookup_failed: 1 | SOAR: created: 0, updated: 0, unchanged: 0, unprojectable: 0, site_lookups: 0, site_lookup_failed: 0

stderr (per-record site-lookup-failed lines, retried on the next sweep):
observation_id='4276100': observed-site lookup unavailable -- using fallback label.

Every existing CalendarEvent, after the takeover sweep:
  facility-url-keyed  : 159 events
  RUN:                : 72 events
  GEM:                : 0 events
  blank-url           : 10 events


In [6]:
print('D-19: the other three key families, before vs. after the takeover sweep:')
for family_name in ('RUN:', 'GEM:', 'blank-url'):
    unchanged = before_families[family_name] == after_families[family_name]
    print(f'  {family_name:<10} unchanged: {unchanged} ({len(before_families[family_name])} events)')
    assert unchanged, f'the takeover sweep must never touch the {family_name} namespace'

changed_titles = [
    (before_key[0], before_key[1], after_families['facility-url-keyed'][pk][1])
    for pk, before_key in before_families['facility-url-keyed'].items()
    if pk in after_families['facility-url-keyed'] and before_key[1] != after_families['facility-url-keyed'][pk][1]
]

# Re-read the routing flag from the environment in this cell rather than trusting cell
# 7022f987's binding -- mirrors the WR-09 pattern already written into cell 250b5d0b.
scratch_routed = bool(os.environ.get('FOMO_DATABASE_PATH'))

# A sweep that narrowed a span (start_time/end_time) or re-linked the companion row
# without changing the title would be missed by the title-only diff above -- compare
# the full snapshot tuple instead so a span-only or link-only change still counts.
changed_keys = [
    pk
    for pk, before_key in before_families['facility-url-keyed'].items()
    if pk in after_families['facility-url-keyed'] and before_key != after_families['facility-url-keyed'][pk]
]

print()
print(
    f'{len(changed_titles)} of {len(before_families["facility-url-keyed"])} pre-existing facility-url-keyed '
    'events were re-titled by the takeover.'
)
print(
    f'{len(changed_keys)} of {len(before_families["facility-url-keyed"])} pre-existing facility-url-keyed '
    'events changed in at least one snapshot field.'
)
print('Sample before -> after title pairs:')
for url, before_title, after_title in changed_titles[:8]:
    print(f'  {url}')
    print(f'    before: {before_title!r}')
    print(f'    after : {after_title!r}')

if scratch_routed:
    assert changed_keys, (
        'No facility-url-keyed event changed in any snapshot field on this scratch-routed run -- '
        'the clone was already swept before this run started. Either re-clone from an un-swept '
        'developer database and re-execute, or, if the developer database itself has legitimately '
        'converged, run this notebook un-routed so the converged branch applies and says so.'
    )
    assert changed_titles, (
        'No facility-url-keyed event was re-titled on this scratch-routed run -- the clone was '
        'already swept before this run started. Either re-clone from an un-swept developer '
        'database and re-execute, or, if the developer database itself has legitimately '
        'converged, run this notebook un-routed so the converged branch applies and says so.'
    )
elif not changed_keys:
    print()
    print(
        "This run found no takeover work -- this commit's takeover cells are historical rather "
        'than a demonstration, and the divergent run lives in the git history of this notebook.'
    )

D-19: the other three key families, before vs. after the takeover sweep:
  RUN:       unchanged: True (72 events)
  GEM:       unchanged: True (0 events)
  blank-url  unchanged: True (10 events)

33 of 159 pre-existing facility-url-keyed events were re-titled by the takeover.
33 of 159 pre-existing facility-url-keyed events changed in at least one snapshot field.
Sample before -> after title pairs:
  https://observe.lco.global/requests/4378323
    before: '[Q] 1m0 11P'
    after : '[O] TFN-1m0 11P'
  https://observe.lco.global/requests/4378331
    before: '[Q] 1m0 11P'
    after : '[O] TFN-1m0 11P'
  https://observe.lco.global/requests/4378332
    before: '[Q] 1m0 11P'
    after : '[S] 1m0 11P'
  https://observe.lco.global/requests/4378333
    before: '[Q] 1m0 11P'
    after : '[S] 1m0 11P'
  https://observe.lco.global/requests/4378334
    before: '[Q] 1m0 11P'
    after : '[S] 1m0 11P'
  https://observe.lco.global/requests/4378335
    before: '[Q] 1m0 11P'
    after : '[S] 1m0 11P'
  

In [7]:
import re

second_sweep_stdout = StringIO()
call_command('project_observation_calendar', stdout=second_sweep_stdout, stderr=StringIO())
second_sweep_summary = second_sweep_stdout.getvalue().strip()

print('First sweep  (the takeover):', first_sweep_summary)
print('Second sweep (unchanged)   :', second_sweep_summary)

# WR-09 (Phase 34 review): second_sweep_summary is the single joined string containing
# both the LCO and the SOAR segments, so a plain substring test above would pass when
# EITHER facility matches -- it would not have caught a non-converged LCO sweep as long
# as the all-zero SOAR segment (true by construction in this database) was present
# somewhere in the string. Split on the facility separator and assert per segment so
# every facility actually in scope is checked independently.
for segment in second_sweep_summary.split(' | ')[1:]:
    for facility_token in ('created: 0', 'updated: 0', 'site_lookups: 0'):
        assert facility_token in segment, f'{facility_token!r} missing from {segment!r}'

# Re-read the routing flag from the environment in this cell, the same way as cell 05528b38.
scratch_routed = bool(os.environ.get('FOMO_DATABASE_PATH'))

# The takeover's size in one number: every created/updated pair across every facility
# segment of the first sweep, summed -- a reader should not have to do this arithmetic.
first_sweep_work = sum(
    int(created) + int(updated)
    for created, updated in re.findall(r'created: (\d+), updated: (\d+)', first_sweep_summary)
)
print()
print(f'First sweep work (created + updated, every facility): {first_sweep_work}')

if scratch_routed:
    assert first_sweep_work > 0, (
        'The first sweep reported zero created+updated work on this scratch-routed run -- the '
        'clone was already swept before this run started. Either re-clone from an un-swept '
        'developer database and re-execute, or, if the developer database itself has legitimately '
        'converged, run this notebook un-routed so the converged branch applies and says so.'
    )
    assert first_sweep_summary != second_sweep_summary, (
        'The first and second sweep summaries are identical on this scratch-routed run -- the '
        'clone was already swept before this run started. Either re-clone from an un-swept '
        'developer database and re-execute, or, if the developer database itself has legitimately '
        'converged, run this notebook un-routed so the converged branch applies and says so.'
    )
elif first_sweep_work == 0:
    print(
        'This run found nothing to take over -- this is the legitimate, already-converged case '
        'the SCHED-06 re-execution in "What happens next" produces, so the convergence claim '
        'below is about a database that was already converged when this run started.'
    )

print()
print(
    "The first sweep's `updated` count includes every record whose title gained its "
    'observed-telescope token during that same run (the one-time site lookup runs '
    'mid-sweep, before the event fields are built) -- so the second sweep reporting '
    'zero created/updated/site_lookups for every facility is convergence, not silence.'
)

First sweep  (the takeover): Done. failed: 0 | LCO: created: 0, updated: 33, unchanged: 126, unprojectable: 0, site_lookups: 14, site_lookup_failed: 1 | SOAR: created: 0, updated: 0, unchanged: 0, unprojectable: 0, site_lookups: 0, site_lookup_failed: 0
Second sweep (unchanged)   : Done. failed: 0 | LCO: created: 0, updated: 0, unchanged: 159, unprojectable: 0, site_lookups: 0, site_lookup_failed: 1 | SOAR: created: 0, updated: 0, unchanged: 0, unprojectable: 0, site_lookups: 0, site_lookup_failed: 0

First sweep work (created + updated, every facility): 33

The first sweep's `updated` count includes every record whose title gained its observed-telescope token during that same run (the one-time site lookup runs mid-sweep, before the event fields are built) -- so the second sweep reporting zero created/updated/site_lookups for every facility is convergence, not silence.


## The whole real corpus, not a sample (ROADMAP criterion 1)


In [8]:
import re

lco_soar_count = ObservationRecord.objects.filter(facility__in=PROJECTED_FACILITIES).count()
facility_event_urls = {key[0] for key in after_families['facility-url-keyed'].values()}
facility_event_count = len(facility_event_urls)

missing_records = [
    record
    for record in ObservationRecord.objects.filter(facility__in=PROJECTED_FACILITIES)
    .select_related('target')
    .order_by('pk')
    if event_url(record, facility_for(record)) not in facility_event_urls
]

# The sweep's own stderr already logged one line per unprojectable row this run --
# "observation_id='<id>' unprojectable (<ExceptionClassName>) -- skipping". Match each
# missing record against that log rather than re-deriving a fresh reason after the fact,
# which could disagree with what the sweep itself actually hit.
unprojectable_reasons = dict(
    re.findall(r"observation_id='([^']+)' unprojectable \(([^)]+)\) -- skipping", first_sweep_stderr.getvalue())
)

print(f'LCO/SOAR ObservationRecord count      : {lco_soar_count}')
print(f'facility-url-keyed CalendarEvent count: {facility_event_count}')
print(f'records with no facility-url-keyed event: {len(missing_records)}')
print(
    f'{lco_soar_count} - {len(missing_records)} = {lco_soar_count - len(missing_records)}; '
    f'matches event count: {lco_soar_count - len(missing_records) == facility_event_count}'
)
assert lco_soar_count - len(missing_records) == facility_event_count

print()
if missing_records:
    print(f'{"observation_id":>14} {"facility":<8} {"status":<12} reason')
    for record in missing_records:
        reason = unprojectable_reasons.get(record.observation_id, 'unprojectable, reason not logged this sweep')
        print(f'{record.observation_id:>14} {record.facility:<8} {record.status:<12} {reason}')
else:
    print('(none -- every LCO/SOAR record in the developer database has its own facility-url-keyed event)')

LCO/SOAR ObservationRecord count      : 159
facility-url-keyed CalendarEvent count: 159
records with no facility-url-keyed event: 0
159 - 0 = 159; matches event count: True

(none -- every LCO/SOAR record in the developer database has its own facility-url-keyed event)


In [9]:
from collections import Counter

marker_tally = Counter()
for key in after_families['facility-url-keyed'].values():
    title = key[1]
    marker = title.split(' ', 1)[0] if title else '(no title)'
    marker_tally[marker] += 1

print('Projected events by stage marker:')
for marker in ('[Q]', '[S]', '[O]', '[X]', '[C]', '[F]', '[?]'):
    print(f'  {marker}: {marker_tally.get(marker, 0)}')
other_markers = {m: c for m, c in marker_tally.items() if m not in ('[Q]', '[S]', '[O]', '[X]', '[C]', '[F]', '[?]')}
if other_markers:
    print('  other (unexpected):', other_markers)
assert sum(marker_tally.values()) == facility_event_count

Projected events by stage marker:
  [Q]: 33
  [S]: 19
  [O]: 74
  [X]: 26
  [C]: 6
  [F]: 1
  [?]: 0


## SCHED-06 baseline: the pending `KEY2026B-004` records (D-20)

`KEY2026B-004`'s still-pending LCO records are the evidence spike 004's PARTIAL verdict
needs to close: over real observing nights, the LCO scheduler places them and the
telescope observes them, `python manage.py updatestatus` refreshes each record from the
portal, and the `post_save` receiver demonstrated above narrows their events -- with no
sweep involved. This section snapshots every pending record now, and writes the same
data to a JSON file beside this notebook so the re-execution described below can diff
against it rather than against a rendered cell.


In [10]:
import json
from collections import Counter
from pathlib import Path

from django.utils import timezone

from solsys_code.observation_projector import stage_for

# WR-09: re-read the environment here rather than relying on the Django setup cell's
# earlier binding -- this guard is the one thing standing between a scratch-routed run
# and clobbering the committed evidence file, so it must be self-contained. Re-running
# just this cell in a fresh kernel (NameError under the old binding) or in a kernel where
# FOMO_DATABASE_PATH was exported after the setup cell already ran (silently takes the
# write branch under the old binding) must not defeat the guard.
SCRATCH_DB_OVERRIDE = os.environ.get('FOMO_DATABASE_PATH') or None

SCHED06_PROPOSAL = 'KEY2026B-004'
SCHED06_BASELINE_PATH = Path.cwd() / 'project_observation_calendar_demo.sched06-baseline.json'

pending_key004 = (
    ObservationRecord.objects.filter(facility='LCO', parameters__proposal=SCHED06_PROPOSAL, status='PENDING')
    .select_related('target')
    .order_by('observation_id')
)

baseline_records = {}
for record in pending_key004:
    facility = facility_for(record)
    url = event_url(record, facility)
    calendar_event = CalendarEvent.objects.filter(url=url).first()
    baseline_records[record.observation_id] = {
        'target': record.target.name,
        'status': record.status,
        'stage': stage_for(record, facility),
        'scheduled_start': None if record.scheduled_start is None else record.scheduled_start.isoformat(),
        'scheduled_end': None if record.scheduled_end is None else record.scheduled_end.isoformat(),
        'event_start': None if calendar_event is None else calendar_event.start_time.isoformat(),
        'event_end': None if calendar_event is None else calendar_event.end_time.isoformat(),
        'event_title': None if calendar_event is None else calendar_event.title,
    }

# D-20/CR-03: the committed SCHED-06 baseline is the evidence UAT Test 4 diffs against. When
# this notebook is routed to a scratch copy (SCRATCH_DB_OVERRIDE set), the baseline file is
# read back but never rewritten -- a copy's snapshot would silently replace the real
# evidence. The two branches do NOT build the same baseline_records (they used to share this
# comment's "identically" claim, which was false): on a scratch-routed run, the copy's rows
# have already been mutated by the takeover sweep cells above, so the committed file's own
# records are substituted in below before the next cell's per-record table renders --
# otherwise that table would present already-swept scratch state under the "SCHED-06
# baseline" heading, contradicting the committed JSON it exists to be diffed against.
if SCRATCH_DB_OVERRIDE is None:
    captured_at = timezone.now().isoformat()
    baseline_payload = {
        'captured_at': captured_at,
        'proposal': SCHED06_PROPOSAL,
        'record_count': len(baseline_records),
        'records': baseline_records,
    }
    with open(SCHED06_BASELINE_PATH, 'w') as fh:
        json.dump(baseline_payload, fh, indent=2)
    print(f'SCHED-06 baseline captured at {captured_at}')
    print(
        f'{len(baseline_records)} pending {SCHED06_PROPOSAL} records; baseline written to {SCHED06_BASELINE_PATH.name}'
    )
else:
    # WR-07: fail with a diagnostic message naming the committed artifact, matching every
    # other precondition in this notebook, rather than a bare FileNotFoundError.
    if not SCHED06_BASELINE_PATH.exists():
        raise RuntimeError(
            f'No committed SCHED-06 baseline at {SCHED06_BASELINE_PATH}. A scratch-routed run reads '
            f'the baseline but never writes it -- run this notebook once un-routed first.'
        )
    with open(SCHED06_BASELINE_PATH) as fh:
        existing_baseline = json.load(fh)
    captured_at = existing_baseline['captured_at']
    # CR-03: show the COMMITTED baseline's own records below, not the scratch copy's
    # already-mutated rows -- keep the copy's rows around only to report their count.
    scratch_records, baseline_records = baseline_records, existing_baseline['records']
    print('Routed to a scratch copy -- showing the COMMITTED baseline below, not this copy.')
    print(f'(the scratch copy currently holds {len(scratch_records)} pending records, already mutated by')
    print('the sweep cells above; they are deliberately not shown -- see the framing above.)')
    print(f'Committed baseline captured_at={captured_at!r}.')

# Recomputed after the branch (not before) so it always reflects whichever baseline_records
# was selected above -- computing it before the branch is what let the scratch-copy tally
# leak out under the old code.
stage_tally = Counter(rec['stage'] for rec in baseline_records.values())
print('Stage tally at baseline:', dict(stage_tally))

Routed to a scratch copy -- showing the COMMITTED baseline below, not this copy.
(the scratch copy currently holds 52 pending records, already mutated by
the sweep cells above; they are deliberately not shown -- see the framing above.)
Committed baseline captured_at='2026-09-11T04:44:59.526430+00:00'.
Stage tally at baseline: {'placed': 18, 'queued': 56}


In [11]:
print(f'{"observation_id":>12} {"target":<24} {"stage":<10} event_start -> event_end')
for observation_id, rec in sorted(baseline_records.items()):
    span = f"{rec['event_start']} -> {rec['event_end']}"
    print(f'{observation_id:>12} {rec["target"]:<24} {rec["stage"]:<10} {span}')

observation_id target                   stage      event_start -> event_end
     4378021 220P                     placed     2026-09-04T16:00:00+00:00 -> 2026-09-04T16:16:50+00:00
     4378022 220P                     placed     2026-09-06T08:14:00+00:00 -> 2026-09-06T08:30:50+00:00
     4378023 220P                     placed     2026-09-08T02:00:00+00:00 -> 2026-09-08T02:16:50+00:00
     4378024 220P                     placed     2026-09-09T18:14:00+00:00 -> 2026-09-09T18:30:50+00:00
     4378025 220P                     queued     2026-09-11T11:14:00+00:00 -> 2026-09-12T16:14:00+00:00
     4378026 220P                     queued     2026-09-13T04:14:00+00:00 -> 2026-09-14T09:14:00+00:00
     4378027 220P                     queued     2026-09-14T21:14:00+00:00 -> 2026-09-16T02:14:00+00:00
     4378028 220P                     queued     2026-09-16T14:14:00+00:00 -> 2026-09-17T19:14:00+00:00
     4378029 220P                     queued     2026-09-18T07:14:00+00:00 -> 2026-09-19T12:

## What happens next

Over the coming nights, run **only**:

```console
$ python manage.py updatestatus
```

Nothing else -- and specifically **not** `project_observation_calendar`. The whole
claim of this section is that a pending `KEY2026B-004` record narrows on the calendar
with nobody running the sweep; running the sweep in between the baseline above and the
re-execution below would quietly prove the opposite of what this section says. Once
several real nights have passed, re-execute this notebook end to end
(`jupyter nbconvert --to notebook --execute --inplace project_observation_calendar_demo.ipynb`)
and commit it again -- the SCHED-06 baseline cell above **overwrites**
`project_observation_calendar_demo.sched06-baseline.json` in place when it re-runs, so
compare the narrowed records with `git diff` on that file (which an un-routed run
overwrites; a scratch-routed run leaves it alone -- see below), not against a copy the
re-execution has already replaced, and the closing table below should be updated to mark
SCHED-06 closed rather than pending. See `34-UAT.md` for the dated table tracking this
follow-up.

When the developer database is holding unrepaired evidence, route a re-execution to a
scratch copy first by exporting an absolute `FOMO_DATABASE_PATH` -- in that case the
baseline file is not rewritten and the `git diff` comparison above has nothing to
compare against. Do that only to refresh this notebook's own narrative; run the real,
un-routed re-execution once the nights being watched have actually elapsed. G-34-2
blocked this section entirely until the receiver was fixed (plan 34-05), so the
committed baseline predates that fix by design.

Any scratch-routed re-execution must clone the database fresh immediately before it
runs, and must never re-use a clone a previous execution already swept -- that is what
produced the empty takeover this section's own evidence cells once showed.


## Requirement -> evidence


In [12]:
closing_table = [
    (
        'PROJ-01',
        'One CalendarEvent per LCO/SOAR record, keyed by the facility URL',
        'cells under "Per-corpus reconciliation"',
    ),
    ('PROJ-02', "The event's span narrows queued -> placed -> observed", '"The receiver, demonstrated" cell'),
    ('PROJ-03', 'Exactly one marker, stage-correct', '"Per-marker tally" cell'),
    (
        'PROJ-05',
        'No-churn, never writes outside its own namespace',
        f'"The one-time takeover" diff/assert cell -- {len(changed_titles)} of '
        f'{len(before_families["facility-url-keyed"])} facility-url-keyed events re-titled this run',
    ),
    (
        'TRIG-01',
        'A post_save trigger, not the observation_change_state hook',
        '"The receiver, demonstrated" cell (schedule-only save)',
    ),
    (
        'TRIG-02',
        'No network call, never raises out of a save',
        '"The receiver, demonstrated" cell (no call_command anywhere in it)',
    ),
    (
        'TRIG-03',
        'The sweep backstops QuerySet.update()/bulk_create() paths',
        f'"The one-time takeover" and second-sweep cells -- first sweep created+updated total '
        f'{first_sweep_work}; second sweep reported all zeros for every facility',
    ),
    (
        'ANNOT-03',
        'sync_lco_observation_calendar retired in favour of the projector',
        'this whole notebook replaces the deleted sync_lco_observation_calendar_demo.ipynb',
    ),
    (
        'SCHED-06',
        'Real KEY2026B-004 records narrow over real time with no sweep',
        "PENDING -- baseline captured above; closed only by this notebook's own re-execution "
        'after real observing nights (see "What happens next")',
    ),
]

print(f'{"Requirement":<10} {"Claim":<62} Evidence')
for req, claim, evidence in closing_table:
    print(f'{req:<10} {claim:<62} {evidence}')

Requirement Claim                                                          Evidence
PROJ-01    One CalendarEvent per LCO/SOAR record, keyed by the facility URL cells under "Per-corpus reconciliation"
PROJ-02    The event's span narrows queued -> placed -> observed          "The receiver, demonstrated" cell
PROJ-03    Exactly one marker, stage-correct                              "Per-marker tally" cell
PROJ-05    No-churn, never writes outside its own namespace               "The one-time takeover" diff/assert cell -- 33 of 159 facility-url-keyed events re-titled this run
TRIG-01    A post_save trigger, not the observation_change_state hook     "The receiver, demonstrated" cell (schedule-only save)
TRIG-02    No network call, never raises out of a save                    "The receiver, demonstrated" cell (no call_command anywhere in it)
TRIG-03    The sweep backstops QuerySet.update()/bulk_create() paths      "The one-time takeover" and second-sweep cells -- first sweep created+updated